In [1]:
from tensorflow.keras.applications import EfficientNetV2B3
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense,Dropout,GlobalAveragePooling2D
from tensorflow.keras.preprocessing import image
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping,ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import classification_report,confusion_matrix
import matplotlib.pyplot as plt
import clahe

I0000 00:00:1792984842.617644    1871 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792984843.450418    1871 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792984847.807005    1871 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
training_path = "../Datasets/data/train/Medical/X-Ray/humerus/Train"
testining_path = "../Datasets/data/test/Medical/X-Ray/humerus/Test"
validation_path = "../Datasets/data/vaild/Medical/X-Ray/humerus"

In [3]:
train_aug = ImageDataGenerator(
    preprocessing_function = lambda img : clahe.clahe_preprocess(img,preprocess_input),
    rotation_range = 10,
    width_shift_range=0.05,
    height_shift_range=0.05,
    zoom_range=0.10,
    shear_range=0.05,
    validation_split = 0.2
)
valid_aug = ImageDataGenerator(
    preprocessing_function=lambda img: clahe.clahe_preprocess(img, preprocess_input),
    validation_split=0.2
)
test_aug = ImageDataGenerator(preprocessing_function = lambda img : clahe.clahe_preprocess(img,preprocess_input))

In [4]:
train_set = train_aug.flow_from_directory(
    training_path,
    target_size = (224,224),
    batch_size=8,
    class_mode="binary",
    seed=42
)
validation_set = train_aug.flow_from_directory(
    validation_path,
    target_size=(224, 224),
    batch_size=8,
    class_mode="binary",
    subset="validation",
    seed=42,
)
test_set = test_aug.flow_from_directory(
    testining_path,
    target_size = (224,224),
    batch_size=8,
    class_mode="binary",
    shuffle = False
)

Found 1436 images belonging to 2 classes.
Found 40 images belonging to 2 classes.
Found 538 images belonging to 2 classes.


In [5]:
base_model = EfficientNetV2B3(weights="imagenet",include_top=False,input_shape=(224,224,3),pooling = 'avg')

I0000 00:00:1792984853.857951    1871 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1763 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2050, pci bus id: 0000:01:00.0, compute capability: 8.6


In [6]:
base_model.trainable = False

In [7]:
x = base_model.output
x = Dense(256,activation="relu")(x)
x = Dropout(0.5)(x)
x = Dense(512,activation='relu')(x)
x = Dropout(0.5)(x)
x = Dense(256,activation='relu')(x)
output = Dense(1,activation='sigmoid')(x)

In [8]:
model = Model(inputs = base_model.input, outputs = output)

In [9]:
model.compile(optimizer='adam',metrics=['accuracy'],loss="binary_crossentropy")

In [10]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)


In [11]:
model.fit(train_set,validation_data=validation_set,epochs=40,callbacks=early_stop)

Epoch 1/40


I0000 00:00:1792984857.985085    1871 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.
I0000 00:00:1792984869.087888    2062 service.cc:153] XLA service 0x7948b403a7a0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1792984869.087928    2062 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 2050, Compute Capability 8.6 (Driver: 13.1.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.24.0)
I0000 00:00:1792984869.635777    2062 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1792984872.721639    2062 cuda_dnn.cc:461] Loaded cuDNN version 92400
I0000 00:00:1792984874.182548    2062 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_28112__.393
I0000 00:00:1792984900.896423    2062 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


 58/180 ━━━━━━━━━━━━━━━━━━━━ 11s 91ms/step - accuracy: 0.5321 - loss: 0.7278

I0000 00:00:1792984911.548843    2062 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_28112__.393


180/180 ━━━━━━━━━━━━━━━━━━━━ 94s 285ms/step - accuracy: 0.6685 - loss: 0.6292 - val_accuracy: 0.8250 - val_loss: 0.4207
Epoch 2/40
180/180 ━━━━━━━━━━━━━━━━━━━━ 16s 87ms/step - accuracy: 0.7326 - loss: 0.5446 - val_accuracy: 0.8500 - val_loss: 0.3966
Epoch 3/40
180/180 ━━━━━━━━━━━━━━━━━━━━ 14s 75ms/step - accuracy: 0.7625 - loss: 0.5127 - val_accuracy: 0.8000 - val_loss: 0.4572
Epoch 4/40
180/180 ━━━━━━━━━━━━━━━━━━━━ 15s 83ms/step - accuracy: 0.7744 - loss: 0.4895 - val_accuracy: 0.8250 - val_loss: 0.3562
Epoch 5/40
180/180 ━━━━━━━━━━━━━━━━━━━━ 15s 82ms/step - accuracy: 0.7758 - loss: 0.4705 - val_accuracy: 0.7750 - val_loss: 0.4267
Epoch 6/40
180/180 ━━━━━━━━━━━━━━━━━━━━ 15s 85ms/step - accuracy: 0.8022 - loss: 0.4572 - val_accuracy: 0.8000 - val_loss: 0.3716
Epoch 7/40
180/180 ━━━━━━━━━━━━━━━━━━━━ 14s 77ms/step - accuracy: 0.8092 - loss: 0.4405 - val_accuracy: 0.7750 - val_loss: 0.4396
Epoch 8/40
180/180 ━━━━━━━━━━━━━━━━━━━━ 15s 84ms/step - accuracy: 0.8050 - loss: 0.4537 - val_accura

In [12]:
prediction_prob = model.predict(test_set)

68/68 ━━━━━━━━━━━━━━━━━━━━ 22s 237ms/step


In [13]:
prediction = (prediction_prob >= 0.5).astype(int)

In [14]:
true_val = test_set.classes


In [15]:
print(classification_report(true_val,prediction))
print(confusion_matrix(true_val,prediction))

              precision    recall  f1-score   support

           0       0.89      0.75      0.82       270
           1       0.78      0.91      0.84       268

    accuracy                           0.83       538
   macro avg       0.84      0.83      0.83       538
weighted avg       0.84      0.83      0.83       538

[[203  67]
 [ 25 243]]


In [16]:
model1 = model

In [17]:
base_model.trainable = True

for layer in base_model.layers[:-40]:
    layer.trainable = False

In [18]:
model.compile(optimizer=Adam(1e-5),loss="binary_crossentropy",metrics=['accuracy'])

In [19]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True
)
reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=2,
    min_lr=1e-7
)

In [20]:
model.fit(train_set,validation_data=validation_set,epochs=30,callbacks=[ early_stop, reduce_lr])

Epoch 1/30


I0000 00:00:1792985305.251615    2062 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_99742__.425


 71/180 ━━━━━━━━━━━━━━━━━━━━ 7s 69ms/step - accuracy: 0.8091 - loss: 0.4522

I0000 00:00:1792985333.158693    2063 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_99742__.425


180/180 ━━━━━━━━━━━━━━━━━━━━ 77s 232ms/step - accuracy: 0.8203 - loss: 0.4417 - val_accuracy: 0.8250 - val_loss: 0.4639 - learning_rate: 1.0000e-05
Epoch 2/30
180/180 ━━━━━━━━━━━━━━━━━━━━ 14s 78ms/step - accuracy: 0.8350 - loss: 0.4152 - val_accuracy: 0.7750 - val_loss: 0.4632 - learning_rate: 1.0000e-05
Epoch 3/30
180/180 ━━━━━━━━━━━━━━━━━━━━ 15s 84ms/step - accuracy: 0.8273 - loss: 0.3991 - val_accuracy: 0.8000 - val_loss: 0.4884 - learning_rate: 1.0000e-05
Epoch 4/30
180/180 ━━━━━━━━━━━━━━━━━━━━ 16s 88ms/step - accuracy: 0.8287 - loss: 0.3951 - val_accuracy: 0.8000 - val_loss: 0.4024 - learning_rate: 1.0000e-05
Epoch 5/30
180/180 ━━━━━━━━━━━━━━━━━━━━ 15s 80ms/step - accuracy: 0.8405 - loss: 0.3709 - val_accuracy: 0.8000 - val_loss: 0.4006 - learning_rate: 1.0000e-05
Epoch 6/30
180/180 ━━━━━━━━━━━━━━━━━━━━ 14s 75ms/step - accuracy: 0.8357 - loss: 0.3598 - val_accuracy: 0.8000 - val_loss: 0.4299 - learning_rate: 1.0000e-05
Epoch 7/30
180/180 ━━━━━━━━━━━━━━━━━━━━ 16s 88ms/step - accura

In [21]:
prob = model.predict(test_set)

68/68 ━━━━━━━━━━━━━━━━━━━━ 15s 125ms/step


In [27]:
pre = (prob >= 0.6).astype(int)

In [28]:
print(classification_report(true_val,pre))
print(confusion_matrix(true_val,pre))

              precision    recall  f1-score   support

           0       0.84      0.81      0.83       270
           1       0.82      0.85      0.83       268

    accuracy                           0.83       538
   macro avg       0.83      0.83      0.83       538
weighted avg       0.83      0.83      0.83       538

[[220  50]
 [ 41 227]]


In [ ]:
model.save("humerus_model.keras")